# Connect4 LA head-to-head benchmark

This notebook benchmarks two `Connect4Lookahead` modules from the local `C4` package.

It uses:
- a **very short Numba cache directory** to avoid Windows path-length/cache issues
- one **warmup** pass before timing



In [1]:
import os
import sys
from pathlib import Path
from tabulate import tabulate


# ---------------- benchmark config ----------------
OLD_MODULE = "C4.fast_connect4_lookahead"
NEW_MODULE = "C4.fast_connect4_lookahead_rollout"
CLASS_NAME = "Connect4Lookahead"

MIN_DEPTH = 1
MAX_DEPTH = 7
GAMES_PER_DEPTH = 10          # total games per equal-depth matchup
RANDOM_OPENING_PLIES = 0      # random legal plies before agents take over
SEED = 666

# ---------------- short Numba cache path ----------------
# This avoids Windows path-length issues when Numba writes cache files for
# nested local functions with long generated filenames.
if os.name == "nt":
    SHORT_NUMBA_CACHE = r"C:\nbc"
else:
    SHORT_NUMBA_CACHE = "/tmp/nbc"

Path(SHORT_NUMBA_CACHE).mkdir(parents=True, exist_ok=True)
os.environ["NUMBA_CACHE_DIR"] = SHORT_NUMBA_CACHE

print("Working dir :", Path.cwd())
print("Numba cache :", SHORT_NUMBA_CACHE)
print("OLD_MODULE  :", OLD_MODULE)
print("NEW_MODULE  :", NEW_MODULE)

Working dir : C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4
Numba cache : C:\nbc
OLD_MODULE  : C4.fast_connect4_lookahead
NEW_MODULE  : C4.fast_connect4_lookahead_rollout


In [2]:
import importlib
import numpy as np
import pandas as pd
import time
from dataclasses import dataclass, asdict

# Fresh import in case the kernel already saw these modules before.
for mod_name in [OLD_MODULE, NEW_MODULE]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

old_mod = importlib.import_module(OLD_MODULE)
new_mod = importlib.import_module(NEW_MODULE)

OldLA = getattr(old_mod, CLASS_NAME)
NewLA = getattr(new_mod, CLASS_NAME)

old_agent = OldLA()
new_agent = NewLA()

print("Loaded OLD:", old_mod.__name__, "from", old_mod.__file__)
print("Loaded NEW:", new_mod.__name__, "from", new_mod.__file__)
print("Class     :", CLASS_NAME)

Loaded OLD: C4.fast_connect4_lookahead from C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\C4\fast_connect4_lookahead.py
Loaded NEW: C4.fast_connect4_lookahead_rollout from C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\C4\fast_connect4_lookahead_rollout.py
Class     : Connect4Lookahead


In [3]:
# Warm up both agents once so JIT compile time does not pollute the benchmark.

def _warmup(agent_obj, depth=1):
    board = np.zeros((6, 7), dtype=np.int8)
    _ = int(agent_obj.n_step_lookahead(board.copy(), 1, depth=depth))
    board[5, 3] = 1
    _ = int(agent_obj.n_step_lookahead(board.copy(), 2, depth=depth))

_warmup(old_agent, depth=max(1, MIN_DEPTH))
_warmup(new_agent, depth=max(1, MIN_DEPTH))

print("Warmup complete.")

Warmup complete.


In [4]:
ROWS, COLS = 6, 7
CENTER_ORDER = [3, 4, 2, 5, 1, 6, 0]

def legal_actions(board: np.ndarray):
    return [c for c in range(COLS) if board[0, c] == 0]

def drop_piece(board: np.ndarray, col: int, mark: int):
    for r in range(ROWS - 1, -1, -1):
        if board[r, col] == 0:
            board[r, col] = mark
            return r
    return -1

def has_four_from(board: np.ndarray, row: int, col: int, token: int) -> bool:
    for dr, dc in ((0,1), (1,0), (1,1), (1,-1)):
        cnt = 1

        rr, cc = row + dr, col + dc
        while 0 <= rr < ROWS and 0 <= cc < COLS and board[rr, cc] == token:
            cnt += 1
            rr += dr
            cc += dc

        rr, cc = row - dr, col - dc
        while 0 <= rr < ROWS and 0 <= cc < COLS and board[rr, cc] == token:
            cnt += 1
            rr -= dr
            cc -= dc

        if cnt >= 4:
            return True
    return False

def random_opening_move(board: np.ndarray, rng: np.random.Generator) -> int:
    legal = legal_actions(board)
    if not legal:
        return 0
    return int(rng.choice(legal))

def choose_move(agent_obj, board: np.ndarray, player_mark: int, depth: int) -> int:
    return int(agent_obj.n_step_lookahead(board.copy(), player_mark, depth=depth))

@dataclass
class GameResult:
    depth_old: int
    depth_new: int
    old_first: bool
    winner: str
    plies: int
    old_moves: int
    new_moves: int
    old_time_s: float
    new_time_s: float
    game_index: int = -1

def play_one_game(old_agent, new_agent, old_depth: int, new_depth: int, old_first: bool,
                  rng: np.random.Generator, random_opening_plies: int = 0) -> GameResult:
    board = np.zeros((ROWS, COLS), dtype=np.int8)

    old_mark = 1 if old_first else 2
    new_mark = 2 if old_first else 1

    side_to_move = 1
    plies = 0

    old_time_s = 0.0
    new_time_s = 0.0
    old_moves = 0
    new_moves = 0

    while True:
        legal = legal_actions(board)
        if not legal:
            return GameResult(old_depth, new_depth, old_first, "draw", plies, old_moves, new_moves, old_time_s, new_time_s)

        # Optional random opening plies to diversify early positions.
        if plies < random_opening_plies:
            c = random_opening_move(board, rng)
        else:
            if side_to_move == old_mark:
                t0 = time.perf_counter()
                c = choose_move(old_agent, board, old_mark, old_depth)
                old_time_s += (time.perf_counter() - t0)
                old_moves += 1
            else:
                t0 = time.perf_counter()
                c = choose_move(new_agent, board, new_mark, new_depth)
                new_time_s += (time.perf_counter() - t0)
                new_moves += 1

        if c not in legal:
            c = legal[0]

        row = drop_piece(board, c, side_to_move)
        plies += 1

        if has_four_from(board, row, c, side_to_move):
            winner = "old" if side_to_move == old_mark else "new"
            return GameResult(old_depth, new_depth, old_first, winner, plies, old_moves, new_moves, old_time_s, new_time_s)

        if np.all(board != 0):
            return GameResult(old_depth, new_depth, old_first, "draw", plies, old_moves, new_moves, old_time_s, new_time_s)

        side_to_move = 2 if side_to_move == 1 else 1

In [5]:
def run_equal_depth_benchmark(old_agent, new_agent, min_depth=1, max_depth=7,
                              games_per_depth=10, random_opening_plies=0, seed=123):
    rng = np.random.default_rng(seed)
    records = []

    for depth in range(min_depth, max_depth + 1):
        print(f"Running depth L{depth} ...")
        for game_idx in range(games_per_depth):
            old_first = (game_idx % 2 == 0)

            result = play_one_game(
                old_agent=old_agent,
                new_agent=new_agent,
                old_depth=depth,
                new_depth=depth,
                old_first=old_first,
                rng=rng,
                random_opening_plies=random_opening_plies,
            )
            result.game_index = game_idx
            records.append(asdict(result))

    return pd.DataFrame(records)

results_df = run_equal_depth_benchmark(
    old_agent=old_agent,
    new_agent=new_agent,
    min_depth=MIN_DEPTH,
    max_depth=MAX_DEPTH,
    games_per_depth=GAMES_PER_DEPTH,
    random_opening_plies=RANDOM_OPENING_PLIES,
    seed=SEED,
)

print(tabulate(results_df, headers="keys", tablefmt="grid", showindex=False, floatfmt=".3f"))

Running depth L1 ...
Running depth L2 ...
Running depth L3 ...
Running depth L4 ...
Running depth L5 ...
Running depth L6 ...
Running depth L7 ...
+-------------+-------------+-------------+----------+---------+-------------+-------------+--------------+--------------+--------------+
|   depth_old |   depth_new | old_first   | winner   |   plies |   old_moves |   new_moves |   old_time_s |   new_time_s |   game_index |
+=============+=============+=============+==========+=========+=============+=============+==============+==============+==============+
|           1 |           1 | True        | old      |      41 |          21 |          20 |        6.932 |        9.465 |            0 |
+-------------+-------------+-------------+----------+---------+-------------+-------------+--------------+--------------+--------------+
|           1 |           1 | False       | new      |      17 |           8 |           9 |        0.000 |        0.001 |            1 |
+-------------+----------

In [6]:
def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for depth, g in df.groupby("depth_old", sort=True):
        total_games = len(g)
        old_wins = int((g["winner"] == "old").sum())
        new_wins = int((g["winner"] == "new").sum())
        draws = int((g["winner"] == "draw").sum())

        old_total_moves = int(g["old_moves"].sum())
        new_total_moves = int(g["new_moves"].sum())

        old_total_time = float(g["old_time_s"].sum())
        new_total_time = float(g["new_time_s"].sum())

        rows.append({
            "depth": int(depth),
            "games": total_games,
            "old_wins": old_wins,
            "new_wins": new_wins,
            "draws": draws,
            "old_win_rate": old_wins / total_games if total_games else np.nan,
            "new_win_rate": new_wins / total_games if total_games else np.nan,
            "draw_rate": draws / total_games if total_games else np.nan,
            "avg_plies": float(g["plies"].mean()),
            "old_avg_ms_per_move": 1000.0 * old_total_time / old_total_moves if old_total_moves else np.nan,
            "new_avg_ms_per_move": 1000.0 * new_total_time / new_total_moves if new_total_moves else np.nan,
            "old_total_time_s": old_total_time,
            "new_total_time_s": new_total_time,
        })

    return pd.DataFrame(rows)

summary_df = summarize_results(results_df)
summary_df

,depth,games,old_wins,new_wins,draws,old_win_rate,new_win_rate,draw_rate,avg_plies,old_avg_ms_per_move,new_avg_ms_per_move,old_total_time_s,new_total_time_s
0,1,10,5,5,0,0.5,0.5,0.0,29.0,47.850879,65.331423,6.938378,9.473056
1,2,10,10,0,0,1.0,0.0,0.0,38.5,0.075788,0.163868,0.014779,0.031135
2,3,10,10,0,0,1.0,0.0,0.0,28.5,0.130733,0.255065,0.018956,0.035709
3,4,10,10,0,0,1.0,0.0,0.0,36.5,0.402783,1.132878,0.074515,0.203918
4,5,10,10,0,0,1.0,0.0,0.0,19.5,0.870248,3.108026,0.087025,0.295263
5,6,10,5,5,0,0.5,0.5,0.0,37.0,2.591739,9.011394,0.479472,1.667108
6,7,10,10,0,0,1.0,0.0,0.0,22.5,4.829206,16.017679,0.555359,1.761945


In [7]:
overall = {
    "games": len(results_df),
    "old_wins": int((results_df["winner"] == "old").sum()),
    "new_wins": int((results_df["winner"] == "new").sum()),
    "draws": int((results_df["winner"] == "draw").sum()),
    "avg_plies": float(results_df["plies"].mean()),
    "old_avg_ms_per_move": 1000.0 * results_df["old_time_s"].sum() / results_df["old_moves"].sum(),
    "new_avg_ms_per_move": 1000.0 * results_df["new_time_s"].sum() / results_df["new_moves"].sum(),
}
pd.DataFrame([overall])

,games,old_wins,new_wins,draws,avg_plies,old_avg_ms_per_move,new_avg_ms_per_move
0,70,60,10,0,30.214286,7.634096,12.888166


In [8]:
results_df.to_csv("la_head_to_head_results.csv", index=False)
summary_df.to_csv("la_head_to_head_summary.csv", index=False)

print("Saved:")
print(" - la_head_to_head_results.csv")
print(" - la_head_to_head_summary.csv")

Saved:
 - la_head_to_head_results.csv
 - la_head_to_head_summary.csv
